# Layer-vortdiag-topo: native isopycnal initialization

Regenerates this experiment's `input/initial_state.nc` and `input/layer_coord.nc`
using its local YAML and generator. This describes the **ten native layers**;
MOM6's **29-cell z-star diagnostic output** is independently configured in
`MOM_input`. Analysis is in `analysis/inspect_anu_tub.ipynb`.


## Experiment-specific configuration

100 × 100 Cartesian cells at 5000 m spacing, ten fixed-temperature isopycnal
layers with reference interfaces 0, 100, 200, 300, 400, 500, 1200, 1900, 2600,
3300 and 4000 m. The YAML below is authoritative. Reference NetCDF files supply
horizontal geometry; their older vertical grid is not the active layer grid.


In [ ]:
from pathlib import Path
import sys
import json
import warnings
import xarray as xr
import matplotlib.pyplot as plt
%matplotlib inline

# Works from this directory or the experiment root; paths inside YAML are config-relative.
folder = Path('/g/data/fu5/users/rd7475/github/phd/idealised_modelling/mom6_sandbox/mom6-eddy-exp1-layer-vortdiag-topo/initialisation')
sys.path.insert(0, str(folder))
from generate_initial_state import load_config, generate, plot_state
config_path = folder / 'config.yaml'
print(config_path.read_text())

## Balance and geometry

The generator uses an exponential velocity profile with 500 m e-folding depth,
evaluated at reference layer centres. The cyclonic eddy displaces isopycnal
interfaces using discrete layer thermal wind. Temperature stays constant in
each layer. Nonzero beta uses the documented centre-f approximate balance.
The printed validation report checks stability, interface ordering and balance.


In [ ]:
# Capture warnings once so the instability is prominent without duplicate round-trip warnings.
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    output_path, report = generate(config_path)
for message in dict.fromkeys(str(item.message) for item in caught):
    print('WARNING:', message)
print('Generated:', output_path)
print(json.dumps(report, indent=2))
(folder / 'validation.json').write_text(json.dumps(report, indent=2) + '\n')

In [ ]:
with xr.open_dataset(output_path, decode_times=False) as state:
    fig = plot_state(state)
    plt.show()

## Interpretation and fresh-start handoff

Inspect the validation report before launching a fresh run. This notebook only
generates inputs; it does not integrate MOM6. Use this experiment's run
instructions in `LOCAL_RUN.md`. Daily native-layer and online-remapped output
are specified in `DIAGNOSTICS.md`; native layers and diagnostic z-star cells
are different coordinates.
